<a href="https://colab.research.google.com/github/affhzhra04/NLP_G7/blob/main/WID3002_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing

In [2]:
# Install required NLP transformer library
!pip install -q sentence-transformers

In [3]:
# Import libraries
import pandas as pd
import numpy as np
import re
import ast
import spacy

# Load spaCy English model for fast preprocessing (disable heavy components to save speed)
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
print("Libraries successfully imported!")

Libraries successfully imported!


In [4]:
# Download the CMU Book Summary Dataset directly
!wget http://www.cs.cmu.edu/~dbamman/data/booksummaries.tar.gz
!tar -xvzf booksummaries.tar.gz
print("Dataset downloaded and extracted successfully!")

--2026-05-18 08:26:02--  http://www.cs.cmu.edu/~dbamman/data/booksummaries.tar.gz
Resolving www.cs.cmu.edu (www.cs.cmu.edu)... 128.2.42.95
Connecting to www.cs.cmu.edu (www.cs.cmu.edu)|128.2.42.95|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16795330 (16M) [application/x-gzip]
Saving to: ‘booksummaries.tar.gz’

booksummaries.tar.g 100%[===================>]  16.02M   705KB/s    in 25s     

2026-05-18 08:26:28 (652 KB/s) - ‘booksummaries.tar.gz’ saved [16795330/16795330]

booksummaries/
booksummaries/README
booksummaries/booksummaries.txt
Dataset downloaded and extracted successfully!


In [5]:
# Read and structure the raw files
# Load book metadata
metadata_cols = ["wiki_id", "freebase_id", "title", "author", "pub_date", "genres", "summary"]
df_meta = pd.read_csv("booksummaries/booksummaries.txt", sep="\t", names=metadata_cols, header=None)

# Keep only the columns critical to your project scope
df = df_meta[["wiki_id", "title", "author", "genres", "summary"]].copy()

# Drop rows that have missing summaries or titles
df = df.dropna(subset=["title", "summary"])

print(f"Total structured books loaded: {len(df)}")
df.head(2)

Total structured books loaded: 16559


,wiki_id,title,author,genres,summary
0,620,Animal Farm,George Orwell,"{""/m/016lj8"": ""Roman \u00e0 clef"", ""/m/06nbt"":...","Old Major, the old boar on the Manor Farm, ca..."
1,843,A Clockwork Orange,Anthony Burgess,"{""/m/06n90"": ""Science Fiction"", ""/m/0l67h"": ""N...","Alex, a teenager living in near-future Englan..."


In [6]:
# Parse genres dictionary into a clean list
def extract_genres(genre_str):
    if pd.isna(genre_str):
        return []
    try:
        # Convert string representation of dict to actual dict
        genre_dict = ast.literal_eval(genre_str)
        return list(genre_dict.values())
    except:
        return []

df["genres_clean"] = df["genres"].apply(extract_genres)

# Drop books that don't have any genre tags (since we need genres for Precision@K)
df = df[df["genres_clean"].map(len) > 0].reset_index(drop=True)

print("Sample parsed genres:", df["genres_clean"].iloc[0])

Sample parsed genres: ['Roman à clef', 'Satire', "Children's literature", 'Speculative fiction', 'Fiction']


In [7]:
# Define Text Cleaning Function using spaCy
def clean_text(text):
    # 1. Lowercase and remove special characters/numbers
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)

    # 2. Use spaCy for tokenization, stop-word removal, and lemmatization
    doc = nlp(text)
    clean_tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_space]

    return " ".join(clean_tokens)

# Test the cleaning pipeline on a dummy string
print("Test cleaning:", clean_text("The fast runners were chasing 3 hidden culprits!"))

Test cleaning: fast runner chase hide culprit


In [8]:
# Process the full summary column
print("Starting text preprocessing... (This may take a moment)")
df["processed_summary"] = df["summary"].apply(clean_text)
print("Preprocessing complete!")

# Preview the difference
print("\n--- Original Summary Snippet ---")
print(df["summary"].iloc[0][:150] + "...")
print("\n--- Cleaned NLP Summary Snippet ---")
print(df["processed_summary"].iloc[0][:150] + "...")

Starting text preprocessing... (This may take a moment)
Preprocessing complete!

--- Original Summary Snippet ---
 Old Major, the old boar on the Manor Farm, calls the animals on the farm for a meeting, where he compares the humans to parasites and teaches the ani...

--- Cleaned NLP Summary Snippet ---
old major old boar manor farm call animal farm meeting compare human parasite teach animal revolutionary song beast england major die young pig snowba...


In [9]:
# Save processed data to a clean CSV file
output_filename = "cmu_books_cleaned.csv"
df[["wiki_id", "title", "author", "genres_clean", "summary", "processed_summary"]].to_csv(output_filename, index=False)

print(f"Phase 1 Complete! Saved clean file as '{output_filename}'")

Phase 1 Complete! Saved clean file as 'cmu_books_cleaned.csv'


# TF-IDF Vectors

In [10]:
# Load the cleaned data from Phase 1
import pandas as pd
import numpy as np
import ast
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
df = pd.read_csv("cmu_books_cleaned.csv")

# Convert the string representation of lists back into actual Python lists
df["genres_clean"] = df["genres_clean"].apply(ast.literal_eval)

# Fill any empty processed summaries with an empty string to prevent vectorizer errors
df["processed_summary"] = df["processed_summary"].fillna("")

print(f"Ready to process {len(df)} books for the ML pipeline!")

Ready to process 12841 books for the ML pipeline!


In [11]:
# Build TF-IDF Feature Matrix
from sklearn.feature_extraction.text import TfidfVectorizer

print("Generating TF-IDF vectors...")
tfidf = TfidfVectorizer(max_features=5000) # Limits vocabulary to top 5000 words for speed
tfidf_matrix = tfidf.fit_transform(df["processed_summary"])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}") # (Number of books, 5000 words)

Generating TF-IDF vectors...
TF-IDF Matrix Shape: (12841, 5000)


# Sentence-BERT

In [12]:
# Build Sentence-BERT Embeddings (FIXED)
from sentence_transformers import SentenceTransformer

print("Loading Pre-trained S-BERT model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding book summaries into high-dimensional vectors... (This may take a few minutes)")

# Combine features to give the model more context clues
df["rich_metadata_text"] = "Title: " + df["title"] + " | Author: " + df["author"] + " | Plot: " + df["summary"]

sbert_embeddings = model.encode(df["rich_metadata_text"].tolist(), show_progress_bar=True)

print(f"S-BERT Embeddings Shape: {sbert_embeddings.shape}") # (Number of books, 384 dimensions)

Loading Pre-trained S-BERT model (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding book summaries into high-dimensional vectors... (This may take a few minutes)


Batches:   0%|          | 0/402 [00:00<?, ?it/s]

S-BERT Embeddings Shape: (12841, 384)


In [13]:
# Core Recommendation Function
def get_recommendations(book_title, k=5, model_type="sbert"):
    # Find index of the book title (case-insensitive)
    idx_list = df[df["title"].str.lower() == book_title.lower()].index
    if len(idx_list) == 0:
        return f"Book '{book_title}' not found in the database."

    idx = idx_list[0]

    # Select which embedding matrix to use
    if model_type == "tfidf":
        query_vector = tfidf_matrix[idx]
        similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    else:
        query_vector = sbert_embeddings[idx].reshape(1, -1)
        similarity_scores = cosine_similarity(query_vector, sbert_embeddings).flatten()

    # Get top K + 1 indices (excluding the input book itself)
    similar_indices = similarity_scores.argsort()[-(k+1):-1][::-1]

    # Return dataframe of recommendations with scores
    results = df.iloc[similar_indices].copy()
    results["similarity_score"] = similarity_scores[similar_indices]
    return results[["title", "author", "genres_clean", "similarity_score"]]

# Quick test run
print("--- Test Recommendation (S-BERT) ---")
try:
    print(get_recommendations(df["title"].iloc[0], k=5, model_type="sbert"))
except Exception as e:
    print("Test failed:", e)

--- Test Recommendation (S-BERT) ---
                               title            author  \
8357               Snowball's Chance         John Reed   
11662          Freddy the Politician  Walter R. Brooks   
11604  Freddy and Simon the Dictator  Walter R. Brooks   
3514                   The Sheep-Pig   Dick King-Smith   
1758     George's Marvelous Medicine        Roald Dahl   

                                        genres_clean  similarity_score  
8357                                        [Parody]          0.778862  
11662           [Comic novel, Children's literature]          0.624969  
11604  [Comic novel, Children's literature, Fiction]          0.588082  
3514                [Children's literature, Fiction]          0.582909  
1758    [Speculative fiction, Children's literature]          0.580865  


# Evaluation

In [14]:
# Precision@K Evaluation Engine
def calculate_precision_at_k(book_title, k=5, model_type="sbert"):
    idx_list = df[df["title"].str.lower() == book_title.lower()].index
    if len(idx_list) == 0:
        return None

    idx = idx_list[0]
    target_genres = set(df["genres_clean"].iloc[idx])

    # Get top K recommendations
    recs = get_recommendations(book_title, k=k, model_type=model_type)
    if isinstance(recs, str): # Error handling if book not found
        return None

    # Calculate hits (Does a recommended book share at least one genre with the target?)
    hits = 0
    for rec_genres in recs["genres_clean"]:
        if len(target_genres.intersection(set(rec_genres))) > 0:
            hits += 1

    # Precision = Hits / K
    return hits / k

print("Evaluation function successfully compiled!")

Evaluation function successfully compiled!


In [15]:
# Comparative Evaluation Execution & Export
import random

# Pick 50 random unique books to test
test_books = random.sample(df["title"].tolist(), min(50, len(df)))

tfidf_precisions = []
sbert_precisions = []

for book in test_books:
    p_tfidf = calculate_precision_at_k(book, k=5, model_type="tfidf")
    p_sbert = calculate_precision_at_k(book, k=5, model_type="sbert")

    if p_tfidf is not None: tfidf_precisions.append(p_tfidf)
    if p_sbert is not None: sbert_precisions.append(p_sbert)

print("--- EVALUATION REPORT ---")
print(f"Average Precision@5 (TF-IDF Baseline): {np.mean(tfidf_precisions):.4f}")
print(f"Average Precision@5 (S-BERT Transformer): {np.mean(sbert_precisions):.4f}")
print("---------------------------------------")

# Save the S-BERT embeddings matrix so Phase 3 UI can read it instantly
np.save("sbert_embeddings.npy", sbert_embeddings)
print("Phase 2 Complete! 'sbert_embeddings.npy' successfully exported.")

--- EVALUATION REPORT ---
Average Precision@5 (TF-IDF Baseline): 0.5600
Average Precision@5 (S-BERT Transformer): 0.6040
---------------------------------------
Phase 2 Complete! 'sbert_embeddings.npy' successfully exported.
